In [30]:
# # 002_dataset_audit.ipynb

# 1. Imports
#       |
#       ↓
# 2. Get WaxalNLP configs
#       |
#       ↓
# 3. Filter _asr configs
#       |
#       ↓
# 4. Load ful_asr
#       |
#       ↓
# 5. Inspect features
#       |
#       ↓
# 6. Inspect samples
#       |
#       ↓
# 7. Create dataset summary

In [31]:
import sys
print(sys.executable)

/workspace/waxal-asr-project/.venv/bin/python


In [32]:
# Import

import os
import pandas as pd
import numpy as np

from datasets import (
    load_dataset,
    get_dataset_config_names,
    Dataset,
    concatenate_datasets
)

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

In [33]:
# Project paths

ROOT_DIR = ".."

DATA_DIR = "../data"

RAW_DIR = "../data/raw"

PROCESSED_DIR = "../data/processed"

print(os.listdir(ROOT_DIR))

['checkpoints', 'requirements.txt', '.git', '.venv', 'data', '.gitignore', 'configs', 'reports', 'Untitled.ipynb', 'src', '.ipynb_checkpoints', 'notebooks', 'experiments']


In [34]:
# Check available WaxalNLP datasets

waxal_configs = get_dataset_config_names(
    "google/WaxalNLP"
)

waxal_configs

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

['ach_asr',
 'ach_tts',
 'aka_asr',
 'amh_asr',
 'bau_tts',
 'dag_asr',
 'dga_asr',
 'ewe_asr',
 'ewe_tts',
 'fat_tts',
 'ful_asr',
 'ful_tts',
 'hau_tts',
 'ibo_tts',
 'kik_tts',
 'kpo_asr',
 'lin_asr',
 'lug_asr',
 'lug_tts',
 'luo_tts',
 'mas_asr',
 'mlg_asr',
 'nyn_asr',
 'nyn_tts',
 'orm_asr',
 'pcm_tts',
 'sid_asr',
 'sna_asr',
 'tir_asr',
 'sog_asr',
 'swa_tts',
 'twi_tts',
 'yor_tts',
 'wal_asr']

In [35]:
# Keep only ASR datasets
waxal_asr_configs = [
    x for x in waxal_configs
    if x.endswith("_asr")
]

waxal_asr_configs

['ach_asr',
 'aka_asr',
 'amh_asr',
 'dag_asr',
 'dga_asr',
 'ewe_asr',
 'ful_asr',
 'kpo_asr',
 'lin_asr',
 'lug_asr',
 'mas_asr',
 'mlg_asr',
 'nyn_asr',
 'orm_asr',
 'sid_asr',
 'sna_asr',
 'tir_asr',
 'sog_asr',
 'wal_asr']

In [3]:
# Fulfulde is a major African language and has ASR data.

fulfulde = load_dataset(
    "google/WaxalNLP",
    "ful_asr",
    # streaming=True
    # streaming=True
)

fulfulde

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/44 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['id', 'speaker_id', 'transcription', 'language', 'gender', 'audio'],
        num_shards: 11
    })
    validation: IterableDataset({
        features: ['id', 'speaker_id', 'transcription', 'language', 'gender', 'audio'],
        num_shards: 2
    })
    test: IterableDataset({
        features: ['id', 'speaker_id', 'transcription', 'language', 'gender', 'audio'],
        num_shards: 2
    })
    unlabeled: IterableDataset({
        features: ['id', 'speaker_id', 'transcription', 'language', 'gender', 'audio'],
        num_shards: 44
    })
})

In [38]:
# Inspect Fulfulde sample
sample = next(iter(fulfulde["train"]))

sample

{'id': 'ful_1896',
 'speaker_id': 'uDlmSsUYZufzajYSbLK3vOcxJsy2',
 'transcription': "P.O.P, be ɗo fauna saare mai be mai. Nda himɓe kugal maajum ɓe ɗo... goɗɗo ɗo dari ha dou, o ɗo yaaɓi hunde dari, o ɗo waɗa kuugal maajum. Ɓe ɗo vo'itina ɓe yiɗi ɓe fauna saare mai, be wada dekoretin maajum be mai.",
 'language': 'ful',
 'gender': '',
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7e18e7cbb8e0>,
 'text': "P.O.P, be ɗo fauna saare mai be mai. Nda himɓe kugal maajum ɓe ɗo... goɗɗo ɗo dari ha dou, o ɗo yaaɓi hunde dari, o ɗo waɗa kuugal maajum. Ɓe ɗo vo'itina ɓe yiɗi ɓe fauna saare mai, be wada dekoretin maajum be mai."}

In [39]:
# Load Nigerian Common Voice
hausa = load_dataset(
    "benjaminogbonna/nigerian_common_voice_dataset",
    "hausa"
)

igbo = load_dataset(
    "benjaminogbonna/nigerian_common_voice_dataset",
    "igbo"
)

yoruba = load_dataset(
    "benjaminogbonna/nigerian_common_voice_dataset",
    "yoruba"
)

In [40]:
# Inspect Common Voice
hausa["train"].features

{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None),
 'client_id': Value('string'),
 'path': Value('string'),
 'sentence': Value('string'),
 'accent': Value('string'),
 'locale': Value('string')}

In [41]:
# Normalize Common Voice
def normalize_common_voice(example, language):

    return {
        "audio": example["audio"],
        "text": example["sentence"],
        "language": language,
        "speaker_id": example["client_id"]
    }

In [42]:
# Apply Normalization

hausa = hausa.map(
    lambda x: normalize_common_voice(x, "hau")
)

igbo = igbo.map(
    lambda x: normalize_common_voice(x, "ibo")
)

yoruba = yoruba.map(
    lambda x: normalize_common_voice(x, "yor")
)


In [43]:
# Remove Unwanted Column

remove_cols = [
    "client_id",
    "path",
    "sentence",
    "accent",
    "locale"
]

hausa = hausa.remove_columns(remove_cols)
igbo = igbo.remove_columns(remove_cols)
yoruba = yoruba.remove_columns(remove_cols)

In [44]:
#  Verify Common Voice format
hausa["train"].features

{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None),
 'text': Value('string'),
 'language': Value('string'),
 'speaker_id': Value('string')}

In [45]:
# Normalize waxal

def normalize_waxal(example):

    return {
        "audio": example["audio"],
        "text": example["transcription"],
        "language": example["language"],
        "speaker_id": example["speaker_id"]
    }

In [50]:
# Apply Waxal normalization

fulfulde = fulfulde.map(
    normalize_waxal
)

In [51]:
# check
fulfulde["train"].features

In [52]:
# Create dataset dictionary

datasets = {
    "hausa": hausa,
    "igbo": igbo,
    "yoruba": yoruba,
    "fulfulde": fulfulde
}


datasets.keys()

dict_keys(['hausa', 'igbo', 'yoruba', 'fulfulde'])

In [53]:
# Dataset size audit for common voice

for name, ds in datasets.items():

    print("\n================")
    print(name)

    for split in ds:

        print(
            split,
            type(ds[split])
        )


hausa
train <class 'datasets.arrow_dataset.Dataset'>
validation <class 'datasets.arrow_dataset.Dataset'>
test <class 'datasets.arrow_dataset.Dataset'>

igbo
train <class 'datasets.arrow_dataset.Dataset'>
validation <class 'datasets.arrow_dataset.Dataset'>
test <class 'datasets.arrow_dataset.Dataset'>

yoruba
train <class 'datasets.arrow_dataset.Dataset'>
validation <class 'datasets.arrow_dataset.Dataset'>
test <class 'datasets.arrow_dataset.Dataset'>

fulfulde
train <class 'datasets.iterable_dataset.IterableDataset'>
validation <class 'datasets.iterable_dataset.IterableDataset'>
test <class 'datasets.iterable_dataset.IterableDataset'>
unlabeled <class 'datasets.iterable_dataset.IterableDataset'>


In [54]:
# Sample transcript audit
for name, ds in datasets.items():

    print("\n================")
    print(name)

    sample = next(iter(ds["train"]))

    print(sample["text"])


hausa
Bude kofar. Na san kina ciki.

igbo
Odeakwụkwọ ọkpụtọrọkpụ ụlọọrụ na-ahụ maka ọrụ ngo na steeti Anambra

yoruba
Ọmọ ẹgbẹ́ òkùnkùn dèrò àtìmọ́lé torí nílùú Ìbàdàn.

fulfulde
P.O.P, be ɗo fauna saare mai be mai. Nda himɓe kugal maajum ɓe ɗo... goɗɗo ɗo dari ha dou, o ɗo yaaɓi hunde dari, o ɗo waɗa kuugal maajum. Ɓe ɗo vo'itina ɓe yiɗi ɓe fauna saare mai, be wada dekoretin maajum be mai.


In [55]:
# Language distribution

language_counts = {}

for name, ds in datasets.items():

    sample = next(iter(ds["train"]))

    language_counts[name] = sample["language"]


language_counts

{'hausa': 'hau', 'igbo': 'ibo', 'yoruba': 'yor', 'fulfulde': 'ful'}

In [56]:
# Language distribution: count Common Voice sizes

common_voice = {
    "hausa": hausa,
    "igbo": igbo,
    "yoruba": yoruba
}


for name, ds in common_voice.items():

    print("\n", name)

    for split in ds:

        print(
            split,
            len(ds[split])
        )


 hausa
train 7206
validation 901
test 901

 igbo
train 4571
validation 571
test 572

 yoruba
train 3336
validation 417
test 418


In [57]:
# Combine Common Voice datasets

common_voice_validation = concatenate_datasets(
    [
        hausa["validation"],
        igbo["validation"],
        yoruba["validation"]
    ]
)


common_voice_test = concatenate_datasets(
    [
        hausa["test"],
        igbo["test"],
        yoruba["test"]
    ]
)

In [ ]:
# Convert Fulfulde first, since it is an iterable and common voice is dataset

fulfulde_train_sample = list(
    fulfulde["train"].take(1000)
)

fulfulde_train_sample[0]

fulfulde_train_sample = Dataset.from_list(
    fulfulde_train_sample
)

#  check

type(fulfulde_train_sample)

In [ ]:
# Combine everything

all_train = concatenate_datasets(
    [
        common_voice_train,
        fulfulde_train_sample
    ]
)

In [ ]:
#  Save audit report

all_train.save_to_disk(
    "../data/processed/multilingual_asr_train"
)